# Headlight glare detector — model comparison

Trains several detectors on **glare_dataset_final** under identical settings and compares them on the **test split**.

**Before running**
1. *Add Input* → your dataset containing the single file `glare_dataset_final_kaggle.bin` (attaches in seconds; the notebook unpacks it).
2. *Settings* → Accelerator **GPU T4 x2** (or P100), **Internet on** (needed to download model weights).
3. Run the cells top to bottom. Each model has its own cell, so you can run them in separate sessions.

**Long runs:** use *Save Version → Save & Run All (Commit)*. It runs in the background (up to 12 h) and keeps everything in `/kaggle/working`.
If a session stops mid-training, re-run the same model cell: it **resumes** from the last checkpoint.

| Model | Approx. time on a T4 at 960 px, 60 epochs |
|---|---|
| YOLO11n | ~1 h |
| YOLOv8s / YOLO11s | ~1.5–2 h each |
| YOLO11m | ~3 h |
| RT-DETR-l | ~4–5 h |
| Saturation baseline | ~1 min (no training) |

In [ ]:
!pip install -q -U ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPUs:", torch.cuda.device_count())
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings and data

In [ ]:
import os, glob, shutil, json, time, yaml
from pathlib import Path

# ---- identical for every model (fair comparison) ----
IMGSZ    = 960     # small headlights (median box ~28 px) need a large input
EPOCHS   = 60      # raise to 100 if you have GPU time
PATIENCE = 15      # early stopping on validation mAP
SEED     = 0
DEVICE   = 0       # one GPU; [0, 1] uses both T4s but multi-GPU is less reliable inside notebooks
CONF_IMG = 0.25    # confidence threshold for the image-level checks (hit rate / false-positive rate)

WORK = Path("/kaggle/working")
DATA = WORK / "data"
RES  = WORK / "results"; RES.mkdir(exist_ok=True)

# Dataset input: either ONE archive file (glare_dataset_final_kaggle.bin - fast to attach) or the unpacked folder.
# It is copied/extracted to a writable place.
if not (DATA / "images" / "train").exists():
    import zipfile
    archives = [p for p in glob.glob("/kaggle/input/**/*", recursive=True)
                if os.path.isfile(p) and p.endswith((".bin", ".zip")) and zipfile.is_zipfile(p)]
    folders = [Path(p).parent for p in glob.glob("/kaggle/input/**/data.yaml", recursive=True)
               if (Path(p).parent / "images" / "train").exists()]
    if archives:
        print("extracting", archives[0]); zipfile.ZipFile(archives[0]).extractall(DATA)
    elif folders:
        print("copying", folders[0]); shutil.copytree(folders[0], DATA, dirs_exist_ok=True)
    else:
        raise FileNotFoundError("Dataset not found: add the dataset with 'Add Input'")
print("dataset ready at", DATA)

cfg = {"path": str(DATA), "train": "images/train", "val": "images/val", "test": "images/test",
       "nc": 1, "names": ["headlight glare"]}
DATA_YAML = WORK / "data_abs.yaml"
yaml.safe_dump(cfg, open(DATA_YAML, "w"), sort_keys=False)
print(open(DATA_YAML).read())

In [ ]:
def label_path(img):
    return str(img).replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"

def has_box(img):
    p = label_path(img)
    return os.path.exists(p) and any(l.strip() for l in open(p))

for s in ("train", "val", "test"):
    imgs = sorted((DATA / "images" / s).glob("*.jpg"))
    g = sum(has_box(i) for i in imgs)
    by_src = {}
    for i in imgs:
        k = i.name.split("_")[0]; by_src[k] = by_src.get(k, 0) + 1
    print(f"{s:5s}: {len(imgs):5d} images | glare {g} ({g/len(imgs):.1%}) | no glare {len(imgs)-g} | {by_src}")

In [ ]:
# test-set subsets: per source (file prefix) for per-source mAP, and glare / no-glare lists for image-level checks
TEST_IMGS    = sorted((DATA / "images" / "test").glob("*.jpg"))
TEST_GLARE   = [str(i) for i in TEST_IMGS if has_box(i)]
TEST_NOGLARE = [str(i) for i in TEST_IMGS if not has_box(i)]
SUBSETS = {}
for prefix in sorted({i.name.split("_")[0] for i in TEST_IMGS}):
    files = [str(i) for i in TEST_IMGS if i.name.startswith(prefix + "_")]
    if not any(has_box(f) for f in files):
        continue                                   # mAP needs at least one box
    lst = WORK / f"test_{prefix}.txt"; lst.write_text("\n".join(files) + "\n")
    y = WORK / f"test_{prefix}.yaml"
    yaml.safe_dump({**cfg, "test": str(lst)}, open(y, "w"), sort_keys=False)
    SUBSETS[prefix] = str(y)
print("per-source test subsets:", list(SUBSETS))
print("test glare images:", len(TEST_GLARE), "| test no-glare images:", len(TEST_NOGLARE))

## 2. Train + evaluate function
For each model: train (or resume), then on the **test split** report mAP50, mAP50-95, precision, recall, speed,
size, per-source mAP50, and two image-level numbers:
- **img_hit_rate**: share of glare test images where the model finds at least one box.
- **img_false_pos_rate**: share of no-glare test images where the model wrongly draws a box (lower is better; for glare suppression this means dimming a scene that did not need it).

In [ ]:
from ultralytics import YOLO, RTDETR

def image_level(model):
    def frac(files):
        hits = 0
        for r in model.predict(files, imgsz=IMGSZ, conf=CONF_IMG, device=DEVICE, stream=True, verbose=False):
            hits += len(r.boxes) > 0
        return hits / max(1, len(files))
    return frac(TEST_GLARE), frac(TEST_NOGLARE)

def run(name, weights, batch, arch=YOLO, **extra):
    out = WORK / "runs" / name
    if not (out / "DONE").exists():
        last = out / "weights" / "last.pt"
        if last.exists():
            print("resuming", name)
            arch(str(last)).train(resume=True)
        else:
            arch(weights).train(data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=batch,
                                seed=SEED, device=DEVICE, workers=4, project=str(WORK / "runs"), name=name,
                                exist_ok=True, plots=True, verbose=False, **extra)
        (out / "DONE").write_text("ok")

    best = arch(str(out / "weights" / "best.pt"))
    res = {"model": name, "batch": batch}
    try:
        from ultralytics.utils.torch_utils import get_flops
        res["params_M"] = round(sum(p.numel() for p in best.model.parameters()) / 1e6, 2)
        res["GFLOPs"] = round(get_flops(best.model, IMGSZ), 1)        # at the training image size
    except Exception as e:
        print("model size unavailable:", e)
    t = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=16, device=DEVICE, plots=True,
                 project=str(WORK / "runs"), name=name + "_test", exist_ok=True, verbose=False)
    res.update(test_mAP50=t.box.map50, test_mAP50_95=t.box.map, test_precision=t.box.mp,
               test_recall=t.box.mr, ms_per_img=t.speed["inference"])
    for prefix, y in SUBSETS.items():
        m = best.val(data=y, split="test", imgsz=IMGSZ, batch=16, device=DEVICE, plots=False, verbose=False,
                     project=str(WORK / "runs"), name=f"{name}_test_{prefix}", exist_ok=True)
        res[f"mAP50_{prefix}"] = m.box.map50
    res["img_hit_rate"], res["img_false_pos_rate"] = image_level(best)
    json.dump(res, open(RES / f"{name}.json", "w"), indent=1)
    print(json.dumps(res, indent=1))
    return res

## 3. Models — priority 1

In [ ]:
r = run("yolov8s", "yolov8s.pt", batch=16)   # earlier baseline family

In [ ]:
r = run("yolo11s", "yolo11s.pt", batch=16)   # current Ultralytics default

## 4. Models — priority 2

In [ ]:
r = run("yolo11n", "yolo11n.pt", batch=32)   # fastest: speed vs accuracy

In [ ]:
r = run("yolo11m", "yolo11m.pt", batch=8)    # bigger: does capacity help small headlights?

## 5. Models — priority 3 (slow)

In [ ]:
r = run("rtdetr_l", "rtdetr-l.pt", batch=4, arch=RTDETR)   # transformer detector; usually needs more epochs

## 6. Non-learned baseline: saturation threshold
Boxes every connected region of near-white saturated pixels (all RGB channels ≥ 235), with blob area as the score.
It shows how much the learned detectors add over "find the brightest spots".
mAP here is computed with the code below (COCO-style 101-point AP), which closely matches Ultralytics' method.

In [ ]:
import cv2, numpy as np
trapz = getattr(np, "trapezoid", None) or np.trapz

def sat_boxes(path, thr=235, min_area=6):
    im = cv2.imread(path)
    core = (im.min(axis=2) >= thr).astype(np.uint8)
    core = cv2.dilate(core, np.ones((5, 5), np.uint8))           # join a lamp's core with its bloom
    n, _, stats, _ = cv2.connectedComponentsWithStats(core, connectivity=8)
    return [([x, y, x + w, y + h], float(a)) for x, y, w, h, a in stats[1:] if a >= min_area]

def gt_boxes(path):
    H, W = cv2.imread(path).shape[:2]; out = []
    for l in open(label_path(path)):
        if l.strip():
            _, cx, cy, w, h = map(float, l.split())
            out.append([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H])
    return out

def iou(b, G):
    ix = np.clip(np.minimum(b[2], G[:, 2]) - np.maximum(b[0], G[:, 0]), 0, None)
    iy = np.clip(np.minimum(b[3], G[:, 3]) - np.maximum(b[1], G[:, 1]), 0, None)
    inter = ix * iy
    return inter / ((b[2] - b[0]) * (b[3] - b[1]) + (G[:, 2] - G[:, 0]) * (G[:, 3] - G[:, 1]) - inter + 1e-9)

def evaluate(preds, gts):
    ths = np.linspace(0.5, 0.95, 10); scores, rows = [], []
    n_gt = sum(len(g) for g in gts.values())
    for k, P in preds.items():
        G = np.array(gts[k], dtype=float).reshape(-1, 4); used = np.zeros((len(ths), len(G)), bool)
        for box, s in sorted(P, key=lambda p: -p[1]):
            row = np.zeros(len(ths), bool)
            if len(G):
                ious = iou(np.array(box, dtype=float), G)
                for ti, t in enumerate(ths):
                    c = np.where((ious >= t) & ~used[ti])[0]
                    if len(c):
                        j = c[np.argmax(ious[c])]; used[ti, j] = True; row[ti] = True
            scores.append(s); rows.append(row)
    if not rows:
        return 0.0, 0.0, 0.0, 0.0
    tp = np.array(rows)[np.argsort(-np.array(scores))]; aps = []
    for ti in range(len(ths)):
        ctp = np.cumsum(tp[:, ti]); cfp = np.cumsum(~tp[:, ti])
        rec = ctp / max(n_gt, 1); prec = ctp / np.maximum(ctp + cfp, 1)
        mpre = np.flip(np.maximum.accumulate(np.flip(np.concatenate([[1.0], prec, [0.0]]))))
        mrec = np.concatenate([[0.0], rec, [1.0]]); x = np.linspace(0, 1, 101)
        aps.append(trapz(np.interp(x, mrec, mpre), x))
    return float(aps[0]), float(np.mean(aps)), float(tp[:, 0].mean()), float(tp[:, 0].sum() / max(n_gt, 1))

t0 = time.time()
preds = {str(i): sat_boxes(str(i)) for i in TEST_IMGS}
ms = (time.time() - t0) * 1000 / len(TEST_IMGS)
gts = {str(i): gt_boxes(str(i)) for i in TEST_IMGS}
m50, m5095, P, R = evaluate(preds, gts)
res = {"model": "saturation_baseline", "params_M": 0, "GFLOPs": 0, "test_mAP50": m50, "test_mAP50_95": m5095,
       "test_precision": P, "test_recall": R, "ms_per_img": ms}
for prefix in SUBSETS:
    ks = [k for k in preds if Path(k).name.startswith(prefix + "_")]
    res[f"mAP50_{prefix}"] = evaluate({k: preds[k] for k in ks}, {k: gts[k] for k in ks})[0]
res["img_hit_rate"] = float(np.mean([len(preds[k]) > 0 for k in TEST_GLARE]))
res["img_false_pos_rate"] = float(np.mean([len(preds[k]) > 0 for k in TEST_NOGLARE]))
json.dump(res, open(RES / "saturation_baseline.json", "w"), indent=1)
print(json.dumps(res, indent=1))

## 7. Comparison table

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = [json.load(open(f)) for f in sorted(RES.glob("*.json"))]
df = pd.DataFrame(rows).set_index("model").sort_values("test_mAP50_95", ascending=False)
cols = ["test_mAP50", "test_mAP50_95", "test_precision", "test_recall", "img_hit_rate", "img_false_pos_rate",
        "ms_per_img", "params_M", "GFLOPs", "batch"] + sorted(c for c in df.columns if c.startswith("mAP50_"))
df = df[[c for c in cols if c in df.columns]]
pd.options.display.float_format = "{:.3f}".format
display(df)
df.to_csv(RES / "comparison.csv")

fig, ax = plt.subplots(figsize=(7, 4.5))
for name, r in df.iterrows():
    ax.scatter(r["ms_per_img"], r["test_mAP50_95"], s=60)
    ax.annotate(name, (r["ms_per_img"], r["test_mAP50_95"]), textcoords="offset points", xytext=(6, 4))
ax.set_xlabel("inference time per image (ms, T4)"); ax.set_ylabel("test mAP50-95")
ax.set_title("Accuracy vs speed — headlight glare detection"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RES / "accuracy_vs_speed.png", dpi=150); plt.show()

## 8. Package results for download

In [ ]:
for d in (WORK / "runs").iterdir():
    if (d / "weights" / "best.pt").exists():
        shutil.copy2(d / "weights" / "best.pt", RES / f"{d.name}_best.pt")
        if (d / "results.csv").exists():
            shutil.copy2(d / "results.csv", RES / f"{d.name}_training_log.csv")
shutil.make_archive(str(WORK / "glare_results"), "zip", root_dir=WORK, base_dir="results")
print("download /kaggle/working/glare_results.zip  (", round(os.path.getsize(WORK / 'glare_results.zip') / 1e6, 1), "MB )")